In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import *
from src.utils import setup_logger, clean_text, save_jsonl
import docling
from docling.document_converter import DocumentConverter
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm

logger = setup_logger()
print("✓ Data ingestion setup complete")

2025-12-07 00:30:05,904 - INFO - PyTorch version 2.5.1+cu121 available.


✓ Data ingestion setup complete


In [2]:
converter = DocumentConverter()

print("Docling converter initialized")
print(f"Supported formats: PDF, DOCX, TXT")
print(f"Raw data directory: {RAW_DATA_DIR}")

Docling converter initialized
Supported formats: PDF, DOCX, TXT
Raw data directory: C:\Users\patel\Documents\chitraksha\data\raw


In [3]:
def process_pdf(pdf_path):
    """Process a single PDF file using Docling."""
    try:
        logger.info(f"Processing: {pdf_path.name}")
        
        result = converter.convert(str(pdf_path))
        text = result.document.export_to_markdown()
        text = clean_text(text)
        
        logger.info(f"Extracted {len(text)} characters from {pdf_path.name}")
        return text
        
    except Exception as e:
        logger.error(f"Error processing {pdf_path.name}: {e}")
        return None

print("PDF processing function defined")

PDF processing function defined


In [4]:
def process_all_pdfs(input_dir, output_dir):
    """Process all PDFs in a directory."""
    pdf_files = list(input_dir.glob("*.pdf"))
    
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return []
    
    print(f"Found {len(pdf_files)} PDF files")
    
    processed_docs = []
    
    for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
        text = process_pdf(pdf_path)
        
        if text:
            output_file = output_dir / f"{pdf_path.stem}.txt"
            output_file.write_text(text, encoding='utf-8')
            
            processed_docs.append({
                "filename": pdf_path.name,
                "text": text,
                "char_count": len(text),
                "source": "pdf"
            })
    
    print(f"\nProcessed {len(processed_docs)} PDFs successfully")
    return processed_docs

print("Batch PDF processing function defined")

Batch PDF processing function defined


In [7]:
def load_mental_health_datasets():
    """Load mental health datasets from multiple sources."""
    all_docs = []
    
    # 1. Load First Hugging Face Dataset - Mental Health Counseling
    try:
        print("Loading HF Dataset 1: Mental Health Counseling Conversations...")
        hf_dataset1 = load_dataset("Amod/mental_health_counseling_conversations")
        
        for item in tqdm(hf_dataset1['train'], desc="Processing HF Dataset 1"):
            text = f"User: {item['Context']}\nCounselor: {item['Response']}"
            
            all_docs.append({
                "text": clean_text(text),
                "char_count": len(text),
                "source": "hf_counseling"
            })
        
        print(f"Loaded {len([d for d in all_docs if d['source'] == 'hf_counseling'])} conversations from HF Dataset 1")
        
    except Exception as e:
        logger.error(f"Error loading HF Dataset 1: {e}")
    
    # 2. Load Second Hugging Face Dataset - Mental Health Chatbot
    try:
        print("\nLoading HF Dataset 2: Mental Health Chatbot Dataset...")
        hf_dataset2 = load_dataset("heliosbrahma/mental_health_chatbot_dataset")
        
        for item in tqdm(hf_dataset2['train'], desc="Processing HF Dataset 2"):
            if 'user_input' in item and 'bot_response' in item:
                text = f"User: {item['user_input']}\nCounselor: {item['bot_response']}"
            elif 'question' in item and 'answer' in item:
                text = f"User: {item['question']}\nCounselor: {item['answer']}"
            elif 'text' in item:
                text = item['text']
            else:
                continue
            
            all_docs.append({
                "text": clean_text(text),
                "char_count": len(text),
                "source": "hf_chatbot"
            })
        
        print(f"Loaded {len([d for d in all_docs if d['source'] == 'hf_chatbot'])} conversations from HF Dataset 2")
        
    except Exception as e:
        logger.error(f"Error loading HF Dataset 2: {e}")
        import traceback
        traceback.print_exc()
    
    # 3. Load Kaggle Dataset (JSON format)
    try:
        print("\nLoading Kaggle dataset...")
        kaggle_file = DATASETS_DIR / "Mental_Health_FAQ.json"
        
        if not kaggle_file.exists():
            print(f"Kaggle JSON not found at: {kaggle_file}")
            print(f"Please download and place it in: {DATASETS_DIR}")
        else:
            import json
            
            with open(kaggle_file, 'r', encoding='utf-8') as f:
                kaggle_data = json.load(f)
            
            if isinstance(kaggle_data, list):
                for item in tqdm(kaggle_data, desc="Processing Kaggle dataset"):
                    question = item.get('question') or item.get('Question') or item.get('questions') or item.get('Questions') or item.get('query')
                    answer = item.get('answer') or item.get('Answer') or item.get('answers') or item.get('Answers') or item.get('response')
                    
                    if question and answer:
                        text = f"User: {question}\nCounselor: {answer}"
                        
                        all_docs.append({
                            "text": clean_text(text),
                            "char_count": len(text),
                            "source": "kaggle_mental_health"
                        })
                
            elif isinstance(kaggle_data, dict):
                if 'intents' in kaggle_data:
                    for intent in tqdm(kaggle_data['intents'], desc="Processing Kaggle dataset"):
                        patterns = intent.get('patterns', [])
                        responses = intent.get('responses', [])
                        
                        for pattern in patterns:
                            for response in responses:
                                text = f"User: {pattern}\nCounselor: {response}"
                                
                                all_docs.append({
                                    "text": clean_text(text),
                                    "char_count": len(text),
                                    "source": "kaggle_mental_health"
                                })
                else:
                    for key, value in tqdm(kaggle_data.items(), desc="Processing Kaggle dataset"):
                        text = f"User: {key}\nCounselor: {value}"
                        
                        all_docs.append({
                            "text": clean_text(text),
                            "char_count": len(text),
                            "source": "kaggle_mental_health"
                        })
            
            kaggle_count = len([d for d in all_docs if d['source'] == 'kaggle_mental_health'])
            print(f"Loaded {kaggle_count} conversations from Kaggle")
    
    except Exception as e:
        logger.error(f"Error loading Kaggle dataset: {e}")
        import traceback
        traceback.print_exc()
    
    print(f"\n{'='*50}")
    print(f"Total documents loaded: {len(all_docs)}")
    print(f"{'='*50}")
    return all_docs

print("Mental health datasets loading function defined")

Mental health datasets loading function defined


In [8]:
print("=== Starting Data Ingestion ===\n")

pdf_docs = process_all_pdfs(RAW_DATA_DIR, PROCESSED_DATA_DIR)

dataset_docs = load_mental_health_datasets()

all_documents = pdf_docs + dataset_docs

print(f"\n=== Ingestion Complete ===")
print(f"Total documents: {len(all_documents)}")
print(f"  - From PDFs: {len(pdf_docs)}")
print(f"  - From Datasets: {len(dataset_docs)}")

metadata_file = PROCESSED_DATA_DIR / "ingestion_metadata.jsonl"
save_jsonl(all_documents, metadata_file)
print(f"\nMetadata saved to: {metadata_file}")

=== Starting Data Ingestion ===

Found 9 PDF files


Processing PDFs:   0%|                                                                                                               | 0/9 [00:00<?, ?it/s]2025-12-07 00:36:04,399 - chitraksha - INFO - Processing: AMHFA_Suicide-and-DSI-Guidelines.pdf
2025-12-07 00:36:04,399 - INFO - Processing: AMHFA_Suicide-and-DSI-Guidelines.pdf
2025-12-07 00:36:04,401 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-07 00:36:04,421 - INFO - Going to convert document batch...
2025-12-07 00:36:04,422 - INFO - Processing document AMHFA_Suicide-and-DSI-Guidelines.pdf
2025-12-07 00:36:06,399 - INFO - Finished converting document AMHFA_Suicide-and-DSI-Guidelines.pdf in 2.00 sec.
2025-12-07 00:36:06,413 - chitraksha - INFO - Extracted 17148 characters from AMHFA_Suicide-and-DSI-Guidelines.pdf
2025-12-07 00:36:06,413 - INFO - Extracted 17148 characters from AMHFA_Suicide-and-DSI-Guidelines.pdf
Processing PDFs:  11%|███████████▍                                                                     


Processed 9 PDFs successfully
Loading HF Dataset 1: Mental Health Counseling Conversations...


Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

2025-12-07 00:37:23,043 - chitraksha - ERROR - Error loading HF Dataset 1: An error occurred while generating the dataset

All the data files must have the same columns, but at some point there are 1 new columns ({'text'}) and 2 missing columns ({'Response', 'Context'}).

This happened while the text dataset builder was generating data using

hf://datasets/Amod/mental_health_counseling_conversations/LICENSE-RAIL-D.txt (at revision d7e86f0813c5690181b41f97403c3674aa55dcef)

Please either edit the data files to have matching columns, or separate them into different configurations (see docs at https://hf.co/docs/hub/datasets-manual-configuration#multiple-configurations)
2025-12-07 00:37:23,043 - ERROR - Error loading HF Dataset 1: An error occurred while generating the dataset

All the data files must have the same columns, but at some point there are 1 new columns ({'text'}) and 2 missing columns ({'Response', 'Context'}).

This happened while the text dataset builder was generating data


Loading HF Dataset 2: Mental Health Chatbot Dataset...


Processing HF Dataset 2: 100%|████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:00<00:00, 20066.21it/s]


Loaded 172 conversations from HF Dataset 2

Loading Kaggle dataset...


Processing Kaggle dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:00<00:00, 31880.70it/s]

Loaded 661 conversations from Kaggle

Total documents loaded: 833

=== Ingestion Complete ===
Total documents: 842
  - From PDFs: 9
  - From Datasets: 833

Metadata saved to: C:\Users\patel\Documents\chitraksha\data\processed\ingestion_metadata.jsonl


In [9]:
# DEBUG CELL: Check what happened with PDFs
import os

print("=== PDF Processing Debug ===\n")

# Check if PDFs exist
pdf_files = list(RAW_DATA_DIR.glob("*.pdf"))
print(f"PDFs in raw folder: {len(pdf_files)}")
for pdf in pdf_files:
    print(f"  - {pdf.name} ({pdf.stat().st_size / 1024 / 1024:.2f} MB)")

# Check if any processed text files exist
txt_files = list(PROCESSED_DATA_DIR.glob("*.txt"))
print(f"\nProcessed text files: {len(txt_files)}")
for txt in txt_files:
    print(f"  - {txt.name} ({txt.stat().st_size / 1024:.2f} KB)")

# Check logs for errors
print("\nChecking logs for PDF processing errors...")
log_file = LOGS_DIR / "chitraksha.log"
if log_file.exists():
    with open(log_file, 'r') as f:
        lines = f.readlines()
        pdf_errors = [line for line in lines[-100:] if 'ERROR' in line and 'pdf' in line.lower()]
        if pdf_errors:
            print("\nRecent PDF errors:")
            for error in pdf_errors[-5:]:
                print(error.strip())
        else:
            print("No recent PDF errors found in logs")

=== PDF Processing Debug ===

PDFs in raw folder: 9
  - AMHFA_Suicide-and-DSI-Guidelines.pdf (0.28 MB)
  - MHFA_Depression-Guidelines.pdf (0.26 MB)
  - MHFA_Non-Suicidal-Self-Injury-Guidelines.pdf (0.17 MB)
  - MHFA_Panic-Attacks-Guidelines.pdf (0.16 MB)
  - MHFA_Psychosis-Guidelines-1.pdf (0.24 MB)
  - MHFA_Psychosis-Guidelines.pdf (0.24 MB)
  - MHFA_Suicidal-Thoughts-and-Behaviours-Guidelines.pdf (0.21 MB)
  - MHFA_Traumatic-Events_Adults-Guidelines.pdf (0.21 MB)
  - PMHP-Basic-Counselling-Skills.pdf (1.73 MB)

Processed text files: 9
  - AMHFA_Suicide-and-DSI-Guidelines.txt (16.75 KB)
  - MHFA_Depression-Guidelines.txt (24.21 KB)
  - MHFA_Non-Suicidal-Self-Injury-Guidelines.txt (15.50 KB)
  - MHFA_Panic-Attacks-Guidelines.txt (9.74 KB)
  - MHFA_Psychosis-Guidelines-1.txt (34.27 KB)
  - MHFA_Psychosis-Guidelines.txt (34.27 KB)
  - MHFA_Suicidal-Thoughts-and-Behaviours-Guidelines.txt (19.09 KB)
  - MHFA_Traumatic-Events_Adults-Guidelines.txt (16.85 KB)
  - PMHP-Basic-Counselling-Skill

In [10]:
if all_documents:
    print("\n=== Sample Documents ===\n")
    
    for i, doc in enumerate(all_documents[:5], 1):
        print(f"Document {i}:")
        print(f"  Source: {doc['source']}")
        print(f"  Length: {doc['char_count']} characters")
        print(f"  Preview: {doc['text'][:200]}...")
        print()
else:
    print("No documents processed")


=== Sample Documents ===

Document 1:
  Source: pdf
  Length: 17148 characters
  Preview: <!-- image --> ## SUICIDAL THOUGHTS &amp; BEHAVIOURS AND DELIBERATE SELF-INJURY ## GUIDELINES FOR PRO VIDING MENT AL HEAL TH FIRST AID TO AN ABORIGINAL OR TORRES STRAIT ISLANDER PERSON <!-- image --> ...

Document 2:
  Source: pdf
  Length: 24652 characters
  Preview: ## DEPRESSION: MENTAL HEALTH FIRST AID GUIDELINES <!-- image --> ## HOW DO I KNOW IF SOMEONE IS EXPERIENCING DEPRESSION? If you notice changes in the personÕs mood, behaviour, energy levels, habits or...

Document 3:
  Source: pdf
  Length: 15770 characters
  Preview: ## NON-SUICIDAL SELF-INJURY: MENTAL HEALTH FIRST AID GUIDELINES <!-- image --> This document includes advice on assisting a person who is injuring themselves, but is not suicidal. It also includes adv...

Document 4:
  Source: pdf
  Length: 9881 characters
  Preview: ## PANIC ATTACKS: MENTAL HEALTH FIRST AID GUIDELINES ## WHAT IS A PANIC ATTACK? A panic attack is an ab

In [11]:
if all_documents:
    total_chars = sum(doc['char_count'] for doc in all_documents)
    avg_chars = total_chars / len(all_documents)
    
    sources = {}
    for doc in all_documents:
        source = doc['source']
        sources[source] = sources.get(source, 0) + 1
    
    print("=== Dataset Statistics ===")
    print(f"Total documents: {len(all_documents):,}")
    print(f"Total characters: {total_chars:,}")
    print(f"Average chars per document: {avg_chars:.0f}")
    print(f"Estimated tokens (÷4): {total_chars // 4:,}")
    
    print("\n=== Documents by Source ===")
    for source, count in sorted(sources.items()):
        percentage = (count / len(all_documents)) * 100
        print(f"  {source}: {count:,} ({percentage:.1f}%)")
    
    print("\n=== Character Distribution ===")
    char_counts = [doc['char_count'] for doc in all_documents]
    print(f"  Minimum: {min(char_counts)} chars")
    print(f"  Maximum: {max(char_counts)} chars")
    print(f"  Median: {sorted(char_counts)[len(char_counts)//2]} chars")

=== Dataset Statistics ===
Total documents: 842
Total characters: 510,252
Average chars per document: 606
Estimated tokens (÷4): 127,563

=== Documents by Source ===
  hf_chatbot: 172 (20.4%)
  kaggle_mental_health: 661 (78.5%)
  pdf: 9 (1.1%)

=== Character Distribution ===
  Minimum: 31 chars
  Maximum: 68759 chars
  Median: 94 chars


In [12]:
print("=== Data Quality Check ===\n")

empty_docs = [doc for doc in all_documents if not doc['text'].strip()]
short_docs = [doc for doc in all_documents if doc['char_count'] < 20]
missing_patterns = [doc for doc in all_documents if 'User:' not in doc['text'] and 'Counselor:' not in doc['text']]

print(f"Empty documents: {len(empty_docs)}")
print(f"Very short documents (<20 chars): {len(short_docs)}")
print(f"Missing User/Counselor pattern: {len(missing_patterns)}")

if empty_docs or short_docs or len(missing_patterns) > len(all_documents) * 0.1:
    print("\n⚠ Warning: Some data quality issues detected")
else:
    print("\n✓ Data quality looks good!")

print(f"\n✓ Valid documents: {len(all_documents) - len(empty_docs) - len(short_docs)}")

=== Data Quality Check ===

Empty documents: 0
Very short documents (<20 chars): 0
Missing User/Counselor pattern: 180

⚠ Warning: Some data quality issues detected

✓ Valid documents: 842


In [13]:
import json

summary = {
    "ingestion_timestamp": pd.Timestamp.now().isoformat(),
    "total_documents": len(all_documents),
    "sources": {
        source: len([d for d in all_documents if d['source'] == source])
        for source in set(doc['source'] for doc in all_documents)
    },
    "total_characters": sum(doc['char_count'] for doc in all_documents),
    "avg_characters": sum(doc['char_count'] for doc in all_documents) / len(all_documents),
}

summary_file = PROCESSED_DATA_DIR / "ingestion_summary.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print("=== Summary Report Saved ===")
print(f"Location: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

=== Summary Report Saved ===
Location: C:\Users\patel\Documents\chitraksha\data\processed\ingestion_summary.json

Summary:
{
  "ingestion_timestamp": "2025-12-07T00:43:34.486696",
  "total_documents": 842,
  "sources": {
    "pdf": 9,
    "kaggle_mental_health": 661,
    "hf_chatbot": 172
  },
  "total_characters": 510252,
  "avg_characters": 606.0
}


In [14]:
print("="*60)
print("PHASE 2: DATA INGESTION COMPLETE")
print("="*60)

print("\n✓ All documents loaded and validated")
print(f"✓ {len(all_documents):,} conversations ready for chunking")
print(f"✓ Metadata saved to: ingestion_metadata.jsonl")
print(f"✓ Summary saved to: ingestion_summary.json")

print("\n" + "="*60)
print("READY FOR PHASE 3: CHUNKING & EMBEDDINGS")
print("="*60)

print("\nNext steps:")
print("1. Save this notebook (Ctrl+S)")
print("2. Create new notebook: 03_chunking_embeddings.ipynb")
print("3. We'll split documents into chunks")
print("4. Generate embeddings for each chunk")
print("5. Prepare data for vector store")

PHASE 2: DATA INGESTION COMPLETE

✓ All documents loaded and validated
✓ 842 conversations ready for chunking
✓ Metadata saved to: ingestion_metadata.jsonl
✓ Summary saved to: ingestion_summary.json

READY FOR PHASE 3: CHUNKING & EMBEDDINGS

Next steps:
1. Save this notebook (Ctrl+S)
2. Create new notebook: 03_chunking_embeddings.ipynb
3. We'll split documents into chunks
4. Generate embeddings for each chunk
5. Prepare data for vector store
